# Recommendation System using Cosine Similarity

## Objective
To build an Anime Recommendation System using Cosine Similarity that recommends similar anime based on genre, type, rating, episodes and popularity.

In [1]:
# Import required libraries

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

## Load Dataset

In [2]:
# Load dataset

df = pd.read_csv("anime.csv")

# Display first 5 rows

df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


## Dataset Information

In [3]:
# Shape of dataset

print("Rows :", df.shape[0])
print("Columns :", df.shape[1])

Rows : 12294
Columns : 7


In [4]:
# Dataset information

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [5]:
# Missing values

df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

## Data Preprocessing

In [6]:
# Fill missing values

df['genre'] = df['genre'].fillna('Unknown')
df['type'] = df['type'].fillna('Unknown')

# Fill rating with median

df['rating'] = df['rating'].fillna(df['rating'].median())

In [7]:
# Check missing values again

df.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

## Feature Engineering

In [8]:
# Convert episodes column

df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')

# Replace NaN episodes with median

df['episodes'] = df['episodes'].fillna(df['episodes'].median())

In [9]:
# Create combined feature

df['combined_features'] = (
    df['genre'].astype(str) + " " +
    df['type'].astype(str)
)

df[['name','combined_features']].head()

,name,combined_features
0,Kimi no Na wa.,"Drama, Romance, School, Supernatural Movie"
1,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili..."
2,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S..."
3,Steins;Gate,"Sci-Fi, Thriller TV"
4,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S..."


## Convert Text Features into Numerical Form

In [10]:
# Vectorize text data

cv = CountVectorizer(stop_words='english')

genre_matrix = cv.fit_transform(df['combined_features'])

genre_matrix.shape

(12294, 52)

## Normalize Numerical Features

In [11]:
# Select numerical features

numeric_features = df[['rating','episodes','members']]

In [12]:
# Normalize features

scaler = MinMaxScaler()

scaled_numeric = scaler.fit_transform(numeric_features)

scaled_numeric.shape

(12294, 3)

In [13]:
# Convert sparse matrix to dataframe

genre_df = pd.DataFrame(
    genre_matrix.toarray()
)

In [14]:
# Numerical dataframe

numeric_df = pd.DataFrame(
    scaled_numeric
)

In [15]:
# Final feature matrix

final_features = pd.concat(
    [genre_df, numeric_df],
    axis=1
)

final_features.shape

(12294, 55)

## Cosine Similarity Matrix

In [16]:
# Compute cosine similarity

cosine_sim = cosine_similarity(final_features)

cosine_sim.shape

(12294, 12294)

## Recommendation Function

In [17]:
def recommend_anime(anime_name,
                    similarity_threshold=0.50,
                    top_n=10):

    anime_name = anime_name.lower()

    matching = df[
        df['name'].str.lower() == anime_name
    ]

    if matching.empty:
        return "Anime not found."

    idx = matching.index[0]

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = []

    for i, score in similarity_scores[1:]:

        if score >= similarity_threshold:

            recommendations.append(
                (df.iloc[i]['name'],
                 round(score,3))
            )

    result = pd.DataFrame(
        recommendations,
        columns=['Anime','Similarity Score']
    )

    return result.head(top_n)

## Generate Recommendations

In [18]:
recommend_anime(
    "Naruto",
    similarity_threshold=0.50,
    top_n=10
)

,Anime,Similarity Score
0,Naruto: Shippuuden,0.998
1,Dragon Ball Z,0.898
2,Dragon Ball Kai,0.885
3,Dragon Ball Super,0.883
4,Medaka Box,0.882
5,Tenjou Tenge,0.882
6,Medaka Box Abnormal,0.881
7,Dragon Ball Kai (2014),0.880
8,Katekyo Hitman Reborn!,0.871
9,Naruto: Shippuuden Movie 4 - The Lost Tower,0.865


In [19]:
recommend_anime(
    "Death Note",
    similarity_threshold=0.60,
    top_n=10
)

,Anime,Similarity Score
0,Higurashi no Naku Koro ni Kai,0.890
1,Mousou Dairinin,0.885
2,Higurashi no Naku Koro ni,0.834
3,Death Note Rewrite,0.803
4,Mirai Nikki (TV),0.801
5,Jigoku Shoujo Mitsuganae,0.790
6,Yakushiji Ryouko no Kaiki Jikenbo,0.781
7,Saint Luminous Jogakuin,0.775
8,Boku dake ga Inai Machi,0.759
9,Zankyou no Terror,0.746


## Experiment with Different Threshold Values

In [20]:
thresholds = [0.3, 0.5, 0.7, 0.9]

for t in thresholds:
    # Temporarily bypass top_n or print the total matching rows
    result = recommend_anime("Naruto", similarity_threshold=t, top_n=len(df))
    print(f"Threshold = {t} -> Total recommendations found: {len(result)}")
    print(result.head(5)) # Still show the top 5 for reference
    print("-" * 30)

Threshold = 0.3 -> Total recommendations found: 4280
                Anime  Similarity Score
0  Naruto: Shippuuden             0.998
1       Dragon Ball Z             0.898
2     Dragon Ball Kai             0.885
3   Dragon Ball Super             0.883
4          Medaka Box             0.882
------------------------------
Threshold = 0.5 -> Total recommendations found: 869
                Anime  Similarity Score
0  Naruto: Shippuuden             0.998
1       Dragon Ball Z             0.898
2     Dragon Ball Kai             0.885
3   Dragon Ball Super             0.883
4          Medaka Box             0.882
------------------------------
Threshold = 0.7 -> Total recommendations found: 100
                Anime  Similarity Score
0  Naruto: Shippuuden             0.998
1       Dragon Ball Z             0.898
2     Dragon Ball Kai             0.885
3   Dragon Ball Super             0.883
4          Medaka Box             0.882
------------------------------
Threshold = 0.9 -> Total recom

## Observations

- Lower threshold values generate more recommendations.
- Higher threshold values generate fewer but more similar recommendations.
- Genre and type contribute significantly to recommendation quality.
- Popularity and rating improve recommendation relevance.
- Threshold 0.3 produced the maximum number of recommendations.
- Threshold 0.9 produced the most restrictive and highly similar recommendations.

## Conclusion

A Content-Based Recommendation System was successfully developed using Cosine Similarity. The system recommends anime with similar genres, type, ratings, episodes, and popularity. Different similarity thresholds were tested to control recommendation quality and quantity.

# Interview Questions
## 1. Difference Between User-Based and Item-Based Collaborative Filtering

User-Based Collaborative Filtering

Finds users with similar preferences.
Recommends items liked by similar users.
User-to-user similarity is calculated.

Item-Based Collaborative Filtering

Finds similar items.
Recommends items similar to those already liked.
Item-to-item similarity is calculated.
Faster and more scalable for large datasets.


## 2. What is Collaborative Filtering?

Collaborative Filtering is a recommendation technique that predicts user interests based on the behavior of similar users.

Steps:

Collect user-item interactions.
Calculate similarity between users or items.
Identify nearest neighbors.
Recommend items that similar users liked.

Examples:

Netflix movie recommendations.
Amazon product recommendations.
Spotify music recommendations.